# The RAG Librarian
#### Project Overview
This notebook implements a Retrieval-Augmented Generation (RAG) system designed to navigate the dense, narrative-heavy texts of classic literature (starting with Victor Hugo and Jane Austen). Unlike standard RAG, which often loses context by "chopping" a story into disconnected pieces, this system uses a "Surgical" approach to reconstruct the narrative surrounding a discovery.

Key Goals:

* Narrative Integrity: Using Neighbor Retrieval to stitch together related plot points.
* Zero-Leakage Search: Using a dedicated LLM Analyst to generate search terms without "memory bleed."
* CPU Optimization: Balancing deep context against local hardware constraints (8GB RAM).

In [ ]:
# --- 1. Standard Utilities ---
import os
import time
import json
import requests
import pypandoc
import numpy as np

# --- 2. Document Loading & Text Processing ---
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- 3. Vector Database & Embeddings ---
import faiss  # Essential for the raw index operations
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- 4. Models & LLM Integration ---
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from langchain_community.llms import Ollama

# --- 5. Global Configuration (The Librarian's Control Panel) ---
# Putting these here allows you to change models in one place
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"
LLM_MODEL_NAME = "llama3.2:1b"

print("✅ All Librarian components are loaded and configured.")

d:\Projects\LLM_RAG_Projects\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All Librarian components are loaded and configured.


In [2]:
# Define the library structure directly in the current folder
library_path = "library"
books = {
    "jane-austen": {
        "url": "https://www.gutenberg.org/ebooks/1342.epub.noimages",
        "filename": "pride-and-prejudice.epub"
    },
    "victor-hugo": {
        "url": "https://www.gutenberg.org/ebooks/135.epub.noimages",
        "filename": "les-miserables.epub"
    }
}

# 2. Create folders and download
for author, info in books.items():
    folder = os.path.join(library_path, author)
    os.makedirs(folder, exist_ok=True)
    
    file_path = os.path.join(folder, info["filename"])
    if not os.path.exists(file_path):
        print(f"📥 Downloading {info['filename']}...")
        response = requests.get(info["url"])
        with open(file_path, 'wb') as f:
            f.write(response.content)
        print(f"✅ Saved to: {file_path}")
    else:
        print(f"📁 {info['filename']} already exists in {folder}")

print("\n📚 Library folders are ready!")

📁 pride-and-prejudice.epub already exists in library\jane-austen
📁 les-miserables.epub already exists in library\victor-hugo

📚 Library folders are ready!


# Chunking
For breaking text into small manageable chunks, we are using the ***/RecursiveCharacterTextSplitter/***, it is "smarter" than a simple character count. It uses a hierarchy of separators to find the best place to cut:

* Paragraphs (\n\n)
* Newlines (\n)
* Spaces ( )
* Characters (Last resort)

It looks for the last space or paragraph break before the 1200-character limit is hit. This ensures that the chunks end on a full word or sentence rather than cutting "Prejudice" into "Preju-" and "-dice."

When we cut a text into chunks, there is a risk that an important piece of information—like a character's name and their action—will be split across the "border" of two chunks, causing the search engine to miss the connection.To solve this, we use a 300-character overlap. This means the last 300 characters of Chunk 1 are repeated at the beginning of Chunk 2.

**Why this matters**:

* ***Semantic Bridges***: It creates a "bridge" between chunks, ensuring that no sentence is lost in the gaps.
* ***Better Retrieval***: If a user's query bridges two chunks, the overlap ensures that the key context is present in both, increasing the mathematical "closeness" (cosine similarity) in the FAISS vector space.
* ***Narrative Continuity***: It prevents the "Cliffhanger Effect," where the AI finds the beginning of a scene but misses the crucial resolution because it was cut off.

In [3]:
# Configuration
library_root = "./library"

# 1. Define the strategy (1200/300 as we agreed)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""] 
)

print("📚 Scanning the library with sequential indexing...")

all_chunks = []
global_chunk_count = 0

# 2. The loop that adds the "neighbor" labels
for author in os.listdir(library_root):
    author_path = os.path.join(library_root, author)
    
    if os.path.isdir(author_path):
        for book_file in os.listdir(author_path):
            if book_file.endswith(".epub"):
                file_path = os.path.join(author_path, book_file)
                print(f"📖 Processing: {book_file} by {author}...")
                
                loader = UnstructuredEPubLoader(file_path)
                docs = loader.load()
                
                # Split the book
                book_chunks = text_splitter.split_documents(docs)
                
                # Tag each chunk with its position
                for i, chunk in enumerate(book_chunks):
                    chunk.metadata["author"] = author
                    chunk.metadata["source"] = book_file
                    chunk.metadata["chunk_index"] = i  
                    chunk.metadata["global_id"] = global_chunk_count 
                    global_chunk_count += 1
                
                all_chunks.extend(book_chunks)

print(f"\n✅ Total Chunks indexed with sequence IDs: {len(all_chunks)}")

📚 Scanning the library with sequential indexing...
📖 Processing: pride-and-prejudice.epub by jane-austen...
📖 Processing: les-miserables.epub by victor-hugo...

✅ Total Chunks indexed with sequence IDs: 4733


# Embedding Strategy — Semantic Vector Space
To transform text into a format the computer can "understand," we use Dense Embeddings. Unlike old-school "Sparse" search (like Ctrl+F), which only looks for exact character matches, Dense embeddings represent text as mathematical coordinates in a high-dimensional vector space.

**The Model**: *BAAI/bge-small-en-v1.5*

**Key Advantages**:
* ***Semantic Intelligence***: It recognizes that "destitution" and "poverty" are conceptually identical. This allows the 'Librarian' to find relevant passages even if your query doesn't use Victor Hugo's exact vocabulary.
* ***High Retrieval Accuracy***: BGE is specifically tuned for retrieval tasks, meaning it is better at distinguishing the "needle" in a large literary "haystack" than general-purpose models.
* ***Efficiency***: It provides a 384-dimensional vector, which is small enough to keep FAISS searches lightning-fast on a CPU while remaining "wide" enough to capture the deep themes of a 1,200-page novel.

In [4]:
# Using a standard Dense Embedding model
# This model is 384-dimensional and very memory efficient
model_name = EMBED_MODEL_NAME
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}

print(f"🧬 Loading Embedding Model: {model_name}...")
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

🧬 Loading Embedding Model: BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1908.37it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Create the FAISS vector store from the chunks
# Define the folder name
index_folder = "faiss_librarian_index"

# 1. Check if the index already exists locally
if os.path.exists(index_folder):
    print(f"📂 Found existing index in '{index_folder}'. Loading now...")
    start_time = time.time()
    
    # allow_dangerous_deserialization=True is required for loading local pickle-based FAISS files
    vector_db = FAISS.load_local(
        index_folder, 
        embeddings, 
        allow_dangerous_deserialization=True
    )
    
    end_time = time.time()
    print(f"✅ Index loaded successfully in {end_time - start_time:.2f} seconds!")

# 2. If it doesn't exist, do the heavy lifting
else:
    print(f"🧠 Index not found. Creating Vector Store from {len(all_chunks)} chunks...")
    start_time = time.time()
    
    # This is the heavy lifting cell
    vector_db = FAISS.from_documents(all_chunks, embeddings)
    
    end_time = time.time()
    print(f"✅ FAISS Index created in {end_time - start_time:.2f} seconds.")
    
    # Save it immediately for next time
    vector_db.save_local(index_folder)
    print(f"💾 Index saved to folder: '{index_folder}'")

🧠 Index not found. Creating Vector Store from 4733 chunks...
✅ FAISS Index created in 741.01 seconds.
💾 Index saved to folder: 'faiss_librarian_index'


## 🔍 The Core Search Engine: Surgical Retrieval & Reranking

To solve the "Fragmented Context" problem inherent in standard RAG—where a literary answer often spans multiple adjacent chunks—we implemented a **Surgical Neighbor Retrieval** system optimized for the T4 GPU.

### Key Architectural Decisions:

* **Step 1: The Keyword Filter (Semantic Guardrail):** Before reranking, the engine applies a "Verbatim Filter." We extract 5 high-signal keywords from the query and scan the initial $k=100$ results. Chunks that do not contain at least one of these specific anchors are discarded. This ensures the Reranker only focuses on "thematic matches" rather than just "vector similarities."

* **Step 2: Window-Based Stitching (Radius=$N$):** For every chunk that passes the keyword filter, the engine "reaches out" to grab the $N$ chunks immediately preceding and following it. This reconstructs the narrative arc that vector search often breaks, providing the LLM with full paragraphs instead of isolated sentences.

* **Step 3: De-duplication:** Overlapping neighbor windows are merged using an index-tracking `set`. This ensures that even if we expand multiple hits, we don't pass redundant tokens to the LLM.

* **Step 4: BGE Reranking:** This allows the Librarian to "re-read" the expanded, keyword-validated strips and prioritize the ones with the highest semantic density ensuring the most relevant narrative arc reaches the top.

In [6]:
# Load the 'Base' Reranker - Optimized for CPU performance
rerank_model_name = RERANK_MODEL_NAME

print(f"🧬 Loading Reranker Model: {rerank_model_name}...")
rerank_tokenizer = AutoTokenizer.from_pretrained(rerank_model_name)
rerank_model = AutoModelForSequenceClassification.from_pretrained(rerank_model_name)
rerank_model.eval()  # Set to evaluation mode for inference

print("✅ Reranker is ready for the Surgical Pipeline.")

🧬 Loading Reranker Model: BAAI/bge-reranker-base...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1588.53it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker is ready for the Surgical Pipeline.


In [7]:
# The Surgical Search Function with Neighbor Retrieval
def surgical_librarian_search(query, keywords, k_initial=40, neighbor_radius=1, top_n=5):
    # 1. Broad Retrieval
    # We lower k_initial to 40 to save memory/time since we are expanding each hit
    initial_docs = vector_db.similarity_search(query, k=k_initial)
    
    # 2. Hard Keyword Filter
    # Only keep docs that mention our specific keywords
    filtered_docs = []
    for doc in initial_docs:
        text_content = doc.page_content.lower()
        if any(word.lower() in text_content for word in keywords):
            filtered_docs.append(doc)
            
    if not filtered_docs:
        return []

    # 3. Neighbor Retrieval & Stitching
    # This is the "Magic" step
    stitched_results = []
    processed_indices = set() # To avoid duplicates if neighbors overlap

    for doc in filtered_docs:
        source_book = doc.metadata.get("source")
        current_idx = doc.metadata.get("chunk_index")
        
        # Create a unique key to track if we've already used this area
        area_key = f"{source_book}_{current_idx}"
        
        if area_key not in processed_indices:
            # Find the range of neighbors
            start_idx = max(0, current_idx - neighbor_radius)
            end_idx = min(len(all_chunks) - 1, current_idx + neighbor_radius)
            
            # Fetch the actual chunks from our master list (all_chunks)
            # We filter by source to ensure we don't bleed into a different book
            neighbor_chunks = [
                c.page_content for c in all_chunks 
                if c.metadata.get("source") == source_book 
                and start_idx <= c.metadata.get("chunk_index") <= end_idx
            ]
            
            # Join them into one larger context block
            full_context = "\n[...] ".join(neighbor_chunks)
            
            stitched_results.append({
                "text": full_context,
                "metadata": doc.metadata
            })
            processed_indices.add(area_key)

    # 4. Reranking (The heavy brain work)
    # We rerank the STITCHED blocks, not just the single chunks
    pairs = [[query, r["text"]] for r in stitched_results]
    
    with torch.no_grad():
        inputs = rerank_tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
        scores = rerank_model(**inputs).logits.view(-1,).float()
    
    # 5. Final Assembly
    final_output = []
    for i, score in enumerate(scores):
        final_output.append({
            "score": score.item(),
            "text": stitched_results[i]["text"],
            "metadata": stitched_results[i]["metadata"]
        })
    
    final_output.sort(key=lambda x: x["score"], reverse=True)
    return final_output[:top_n]

print("🚀 Surgical Search (with Neighbor Retrieval) is ready.")

🚀 Surgical Search (with Neighbor Retrieval) is ready.


In [8]:
# Testing the Surgical Search with a specific query and keywords related to "Les Misérables"

# 1. Define the test query and specific keywords
test_query = "Describe the physical appearance of Jean Valjean when he first enters the inn in Digne."
test_keywords = ["Digne", "passport", "yellow", "iron-shod", "knapsack", "cap"]

print(f"🔍 Running Surgical Search for: {test_query}")

# 2. Execute the search
# This pulls 100 from FAISS, filters by keywords, and reranks with BGE-Base
search_results = surgical_librarian_search(test_query, test_keywords, k_initial=150, neighbor_radius=2, top_n=5)

# 3. Display the survivors
if not search_results:
    print("❌ No matches found. Check if the keywords exist in the book text.")
else:
    for i, res in enumerate(search_results):
        print(f"Rank {i+1} (Reranker Score: {res['score']:.4f})")
        print(f"Content: {res['text'][:400]}...") # Showing first 400 chars
        print("-" * 30)

🔍 Running Surgical Search for: Describe the physical appearance of Jean Valjean when he first enters the inn in Digne.
Rank 1 (Reranker Score: 0.9742)
Content: . It was evident that the person who had had the ordering of that unclean procession had not classified them. These beings had been fettered and coupled pell-mell, in alphabetical disorder, probably, and loaded hap-hazard on those carts. Nevertheless, horrors, when grouped together, always end by evolving a result; all additions of wretched men give a sum total, each chain exhaled a common soul, a...
------------------------------
Rank 2 (Reranker Score: 0.5953)
Content: CHAPTER III—THE HEROISM OF PASSIVE OBEDIENCE.

The door opened.

It opened wide with a rapid movement, as though some one had given it an energetic and resolute push.

A man entered.

We already know the man. It was the wayfarer whom we have seen wandering about in search of shelter.

He entered, advanced a step, and halted, leaving the door open behind him. He 

## The LLM Keyword Analyst
### Isolated Keyword Analysis
A major challenge with small models (Llama 3.2:1b) is context leakage, where the model "hallucinates" keywords from previous search queries.

#### Architecture:

* ***Strict Isolation***: We use a hardened system prompt and triple-quote delimiters to "wall off" the current query.
* ***Temperature Tuning***: Set to 0.1 to maximize determinism and prevent the model from wandering into its training data or conversation history.
* ***JSON Enforcement***: We force the model to output a strictly formatted JSON list to ensure programmatic compatibility with our search function.

In [9]:
# Define the function to generate the keywords using Llama 3.2:1b
def get_llm_keywords(query):
    """
    Uses Llama 3.2:1b with strict isolation to prevent context leakage.
    """
    print(f"🧠 Analyst is extracting keywords (Isolated Mode)...")
    
    # SYSTEM PROMPT: Forces the model to ignore everything outside the specific task
    system_prompt = (
        "TASK: Extract keywords from the text below. "
        "CONSTRAINT 1: Ignore all previous conversations or topics. "
        "CONSTRAINT 2: Only use words found in or directly relevant to the current input. "
        "CONSTRAINT 3: Do not mention 'Digne' or 'knapsack' unless they are in the current input. "
        "OUTPUT: JSON list only."
    )
    
    # USER PROMPT: Uses triple quotes to "wall off" the query
    user_prompt = f"""
    ### INPUT QUERY:
    \"\"\"{query}\"\"\"
    
    ### INSTRUCTION:
    Identify 5 unique nouns/entities from the INPUT QUERY above for a search engine. 
    Return as JSON: {{"keywords": ["word1", "word2"]}}
    """

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": LLM_MODEL_NAME,
                "system": system_prompt,
                "prompt": user_prompt,
                "stream": False,
                "format": "json",
                "options": {
                    "temperature": 0.1,  # Lower temperature = less "creativity" / more focus
                    "num_predict": 50    # Stop the model from rambling
                }
            }
        )
        response.raise_for_status()
        
        result = json.loads(response.json()['response'])
        keywords = result.get('keywords', [])
        
        print(f"✨ Keywords Generated: {keywords}")
        return keywords

    except Exception as e:
        print(f"⚠️ Keyword generation failed: {e}")
        return [word.strip(",.?!") for word in query.split() if len(word) > 3]

In [10]:
# Test the Keyword Analyst
test_query = "What was Gavroche doing when he was singing and collecting bullets from the dead soldiers?"

# Call the function
generated_keywords = get_llm_keywords(test_query)

print(f"\n📝 Original Query: {test_query}")
print(f"🔑 Generated Keywords: {generated_keywords}")

🧠 Analyst is extracting keywords (Isolated Mode)...
✨ Keywords Generated: ['Gavroche', 'singing', 'collecting', 'bullets', 'soldiers']

📝 Original Query: What was Gavroche doing when he was singing and collecting bullets from the dead soldiers?
🔑 Generated Keywords: ['Gavroche', 'singing', 'collecting', 'bullets', 'soldiers']


# Synthesis & The Interactive Interface
## Grounded Answer Generation
The final stage of the pipeline is Synthesis. Once the "Surgical Search" has retrieved the most relevant narrative strips, we use the LLM to weave them into a coherent answer.

**Key Design Principles**:

* ***Context Grounding***: The model is strictly instructed to act as a "Witness." It must answer using only the provided source text. If the information is missing from the retrieved chunks, the model is trained to admit it rather than hallucinate based on its internal training data.
* ***Source Attribution***: The system is prompted to cite specific "Sources" (e.g., Source 1, Source 2) so the user can verify the librarian's claims against the original text.

In [11]:
# Define the function to synthesize the final answer using Llama 3.2:1b
def synthesize_answer(query, search_results):
    """
    Takes the top search results and uses Llama 3.2:1b to write a final answer.
    """
    print(f"✍️ Synthesizing final answer...")
    
    # 1. Combine the top results into a single context block
    context_text = ""
    for i, res in enumerate(search_results):
        context_text += f"--- SOURCE {i+1} ---\n{res['text']}\n\n"

    # 2. Prepare the prompt for Ollama
    system_prompt = (
        "You are the 'Modular Librarian'. Your job is to answer questions based ONLY on the provided source text. "
        "If the answer is not in the text, say you do not have enough information. "
        "Be descriptive and maintain a literary tone."
    )
    
    user_prompt = f"""
    ### CONTEXT:
    {context_text}
    
    ### USER QUESTION:
    "{query}"
    
    ### INSTRUCTIONS:
    Using the sources above, provide a detailed answer to the question. 
    Cite which Source you are using if possible.
    """

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": LLM_MODEL_NAME,
                "system": system_prompt,
                "prompt": user_prompt,
                "stream": False,
                "options": {
                    "temperature": 0.3 # Slightly higher for better prose, but still focused
                }
            }
        )
        response.raise_for_status()
        return response.json()['response']

    except Exception as e:
        return f"⚠️ Failed to synthesize answer: {e}"

In [12]:
# Define the function that will run the entire pipeline
def ask_the_librarian(query):
    """
    The engine of the pipeline. It takes a query and runs the 3-step RAG process.
    """
    print(f"\n🚀 Researching: '{query}'")
    start_all = time.time()

    # Step 1: LLM Keyword Extraction
    keywords = get_llm_keywords(query)
    
    # Step 2: Surgical Search (FAISS + Filter + Stitch + Rerank)
    # Using your optimized settings: k=100, radius=2
    search_results = surgical_librarian_search(
        query, 
        keywords, 
        k_initial=100, 
        neighbor_radius=2, 
        top_n=5
    )

    # Step 3: Synthesis
    if not search_results:
        print("\n❌ The Librarian couldn't find any specific evidence in the text.")
    else:
        answer = synthesize_answer(query, search_results)
        
        print("\n" + "="*60)
        print("📖 THE LIBRARIAN'S ANSWER:")
        print("="*60)
        print(answer)
        print("="*60)
    
    end_all = time.time()
    print(f"⏱️ Total processing time: {end_all - start_all:.2f} seconds.")

# The Interactive Interface
## The Librarian's Desk
This final loop combines the Analyst, the Surgical Search, and the Synthesizer into a single interactive pipeline. It allows for multi-turn research within the library, enabling a user to query the engine through a continuous terminal-style interface.


**Note** : *The Librarian is stateless which means it will not remember your previous question and the answer. If you want to follow up you need to frame you question properly.*

In [13]:
# Continuous interactive loop for user questions
print("📚 The Austen & Hugo Library is now Open.")
print("Type 'quit' or 'exit' to leave the library.\n")

while True:
    # 1. Get user input
    user_input = input("🧐 What would you like to research today?: ")
    
    # 2. Check if the user wants to stop
    if user_input.lower() in ["exit", "quit", "q"]:
        print("👋 Goodbye!")
        break
    
    # 3. Skip empty inputs
    if not user_input.strip():
        print("⚠️ Please ask a specific question.")
        continue
        
    # 4. Run the RAG pipeline
    try:
        # Calling the function with the user input
        ask_the_librarian(user_input)
        print("\n" + "-"*40 + "\n") # Visual separator between questions
    except Exception as e:
        print(f"❌ The Librarian is confused: {e}")

📚 The Austen & Hugo Library is now Open.
Type 'quit' or 'exit' to leave the library.




🚀 Researching: 'Describe how Jean Valjean manages to climb the high wall to escape the soldiers, and what he uses to help Cosette get over it.'
🧠 Analyst is extracting keywords (Isolated Mode)...
✨ Keywords Generated: ['wall', 'soldiers', 'escape', 'Cosette', 'Jean Valjean']
✍️ Synthesizing final answer...

📖 THE LIBRARIAN'S ANSWER:
According to the text, Jean Valjean manages to climb the high wall to escape the soldiers by using a ladder. He is described as "calculating that if the men were still following him, he could not fail to get a good look at them, as they traversed this illuminated space" (Source 5). This suggests that he uses the moonlight to his advantage, allowing him to observe the soldiers' movements and plan his escape.

As for how he helps Cosette get over the wall, the text does not provide a detailed description. However, it does mention that he "made haste to quit the Rue Pontoise" (Source 5), suggesting that he may have helped Cosette escape from the area where th

## Conclusion
### Project Summary
The "RAG Librarian" successfully solves the problem of "context fragmentation" in literary RAG. By leveraging neighbor retrieval, we are able to provide Llama 3.2:1b with enough narrative surrounding a hit to answer complex, descriptive questions that a standard vector search would miss.

**Performance Note**: *On local CPU hardware, the "Pre-fill" stage (where the LLM reads the context) is the primary latency bottleneck. We found that providing 5 sources at Radius 2 (~30,000 characters) creates the most accurate literary descriptions, though it requires a higher processing time compared to shorter, "snackable" RAG snippets.*



## Conclusion & Final Observations
### Project Summary
The "RAG Librarian" successfully demonstrates that a Surgical RAG architecture can navigate the dense, interconnected narratives of classic literature. By leveraging Neighbor Retrieval, we bridged the gap between disconnected text chunks, allowing the system to reconstruct complex scenes that a standard vector search would fail to capture.

### Test Results & Analysis
During our final evaluation, the system displayed a clear "Success vs. Reasoning" trade-off:

The Hugo Test (The Escape): The search engine correctly identified the atmospheric details (moonlight and specific street names), but the **Llama 3.2:1b model** *hallucinated* a "ladder" for the escape. This suggests that while retrieval was accurate, a 1B model may "fill the gaps" with common logic when the specific narrative mechanism (the lamp-cord) isn't explicitly dominant in the top-ranked chunks.

The Austen Test (The Proposal): The system performed exceptionally well at capturing social subtext and character motivation (Mr. Collins's "female delicacy" argument). However, it struggled with complex legal concepts like "land entails," substituting a fictional family history to explain the inheritance.

#### **Performance & Hardware Constraints**
* ***The Bottleneck***: On local CPU hardware (8GB RAM), the Pre-fill stage is the primary latency driver. Processing 30,000 characters of context creates a ~6-minute wait time. 
* ***The "Brain" Ceiling***: Our findings confirm that while the Surgical Pipeline (Retrieval + Reranking) is robust, a 1B-parameter model lacks the "literary memory" to avoid all hallucinations.